# **3. Baseline Model**

In [2]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sidetable
import sklearn
import feature_engine
import scipy
from scipy import stats
from pathlib import Path
import pickle

%matplotlib inline
sns.set_style('darkgrid')


In [3]:
# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [4]:
# define file path for processed data
parent_path = Path.cwd().parent
file_path = parent_path.joinpath("models", "processed.pkl")

In [5]:
# Load the processed data from the previous step
print(f"Loading processed data...")
with open(file_path, "rb") as f:
    processed_data = pickle.load(f)

Loading processed data...


In [6]:
processed_data.keys()

dict_keys(['X_train', 'y_train', 'X_test', 'test_id', 'numeric_features', 'ordinal_features', 'categorical_features', 'boolean_features', 'year_features'])

In [7]:
# Extract features and target variable
X_train = processed_data["X_train"]
y_train = processed_data["y_train"]
X_test = processed_data["X_test"]
numeric_feat = processed_data["numeric_features"]
ordinal_feat = processed_data["ordinal_features"]
categorical_feat = processed_data["categorical_features"]
bool_feat = processed_data["boolean_features"]
year_feat = processed_data["year_features"]

In [8]:
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

Training data shape: (1459, 23)
Test data shape: (1457, 23)


#### **1. Feature Selection**

In [27]:
# 1. Feature Selection for Linear Models
print("\n=== FEATURE SELECTION ===")

# Create dictionary to store different feature sets
feature_sets = {}


=== FEATURE SELECTION ===


**Preference 1.1. Correlation Analysis: compute correlation between numeric features and the target variable**


In [22]:
correlations = X_train[ordinal_feat + numeric_feat + year_feat + bool_feat].corrwith(y_train).abs().sort_values(ascending=False)
print("Top correlated features with the target variable:")
print(correlations)

Top correlated features with the target variable:
LivAreaQual                0.795007
OverallQual                0.793055
TotalSF                    0.765099
NeighborhoodMedianPrice    0.720926
ExterQual                  0.684381
KitchenQual                0.660666
GarageCars                 0.640954
TotalBath                  0.633484
BsmtQual                   0.585748
HouseAge                   0.573607
QualCond                   0.565831
GarageFinish               0.549590
RemodeledAge               0.513813
GarageYrBlt                0.508240
HasFireplace               0.472022
Fireplaces                 0.466967
HasPorch                   0.413045
MasVnrArea                 0.405516
TotalPorchSF               0.337434
CentralAir                 0.251326
HasGarage                  0.236829
BsmtFinSF1                 0.185520
dtype: float64


**Prefrence 2: Use Ridge Regression to compute feature importance**


In [18]:
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Transformes to cast boolean to int
bool_to_int = FunctionTransformer(lambda x: x.astype(int))

# Define preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_feat),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_feat),
        ("boolean", bool_to_int, bool_feat),
        ("ordinal", "passthrough", ordinal_feat),
        ("year", "passthrough", year_feat)
    ]
)

# build pipeline
ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=10.0))
])
ridge_pipeline.fit(X_train, y_train)

# Extract features names after one-hot encoding
features_name = (
    numeric_feat +
    list(ridge_pipeline.named_steps['preprocessor'].named_transformers_['categorical'].get_feature_names_out(categorical_feat)) +
    bool_feat + ordinal_feat + year_feat
) 

# Extract coefficients from the Ridge model
ridge_coef = ridge_pipeline.named_steps['ridge'].coef_

feature_importance = pd.DataFrame({
    "Feature": features_name,
    "Importance": ridge_coef
})
feature_importance.sort_values(by="Importance", ascending=False, inplace=True, ignore_index=True)
print("Top 20 features by importance from Ridge Regression:")
display(feature_importance.head(20))


Top 20 features by importance from Ridge Regression:


,Feature,Importance
0,Neighborhood_NoRidge,27612.306051
1,TotalSF,20340.434725
2,Neighborhood_StoneBr,18199.480640
3,GarageCars,15816.627446
4,Fireplaces,14545.714476
5,Neighborhood_BrkSide,13638.128931
6,NeighborhoodMedianPrice,13187.580203
7,Neighborhood_MeadowV,11110.197771
8,Neighborhood_NridgHt,10705.531313
9,Neighborhood_IDOTRR,10599.184349


In [28]:
# Feature Set 2: Top features based on Ridge importance
# Get top 20 features from ridge importance
top_ridge_features = feature_importance['Feature'][:20].tolist()
top_ridge_features

['Neighborhood_NoRidge',
 'TotalSF',
 'Neighborhood_StoneBr',
 'GarageCars',
 'Fireplaces',
 'Neighborhood_BrkSide',
 'NeighborhoodMedianPrice',
 'Neighborhood_MeadowV',
 'Neighborhood_NridgHt',
 'Neighborhood_IDOTRR',
 'QualCond',
 'KitchenQual',
 'ExterQual',
 'TotalBath',
 'OverallQual',
 'BsmtFinSF1',
 'Neighborhood_Edwards',
 'LivAreaQual',
 'GarageFinish',
 'Neighborhood_Sawyer']